# 04 — Multivariate anomaly detection and diagnosis

Detectors behind one `Detector` interface, evaluated by the same code against known events:
PCA with Hotelling T² / SPE (F, Box, KDE and SPOT limits), dynamic PCA, moving-window PCA (RPCA) and
the author's **MSCVAE** (through `anomaly/mscvae_adapter.py`).

* **Hourly** model (fast events: blackouts, ≥ 1 GW load interruptions): load, generation by source and
  interchange per subsystem. References: two normal windows around the 2023-04 MMGD break.
* **Daily** model (slow anomalies: water crises, COVID, strike): daily means + EAR and ENA. Reference
  2016–2019.

Every fit uses the reference period only; monitoring runs from 2010 to the last day. An alarm needs
*m of the last n* points above the limit. Detectors are compared at their own limits **and** at a
matched 5% false-alarm rate.

> Uses torch only for MSCVAE (no LightGBM here). `RUN_MSCVAE = False` skips it (~30–60 min on an M4).

In [ ]:
%matplotlib inline
import pandas as pd

from ons_energy.anomaly.deseason import WeeklyBaselineResidualizer
from ons_energy.anomaly.dpca import select_lags
from ons_energy.anomaly.experiment import matched_operating_point, mspc_config, run_mspc
from ons_energy.features.mspc import daily_matrix, hourly_matrix
from ons_energy.tracking import log_mspc
from ons_energy.viz import mspc_plots as P

pd.set_option("display.precision", 3)
LOG_MLFLOW = True
RUN_MSCVAE = True
SUMMARY_COLS = ["detector", "preprocess", "n_vars", "k", "ref_alarm_rate", "far", "far_raw", "detection_rate",
                "median_delay_h", "n_events", "top_subsystem_match"]


def run_all(cfgs):
    out = {}
    for c in cfgs:
        out[c.name] = run_mspc(c)
        if LOG_MLFLOW:
            log_mspc(c, out[c.name])
        s = out[c.name]["summary"]
        print(f"{c.name:<26} FAR={s['far']:.3f}  detection={s['detection_rate']:.3f}")
    return out


def summary_table(results):
    s = pd.DataFrame([r["summary"] for r in results.values()])
    return s[[c for c in SUMMARY_COLS if c in s.columns]].sort_values("far")


def matched_table(cfgs, results, target_far=0.05):
    rows = [{"detector": results[c.name]["summary"]["detector"], **matched_operating_point(c, results[c.name]["scores"], target_far)} for c in cfgs]
    return pd.DataFrame(rows).sort_values("detection_rate", ascending=False)

## Hourly scale

### DPCA lag selection
Lag-1 autocorrelation of T² on the reference for 0–3 lags (lower = whiter scores).

In [ ]:
X = hourly_matrix("2022-05-01", "2023-04-28").drop(columns=["SE_wind"])
R = WeeklyBaselineResidualizer(hourly=True).transform(X).dropna().loc["2022-06-01":]
select_lags([R], max_lags=3, var_explained=0.9)

### Experiments
Preprocessing: `raw` levels or causal `weekly_baseline` residuals (x_t minus the mean of the same hour
over the previous 4 weeks).

In [ ]:
PCA = {"var_explained": 0.9, "alpha": 0.01}
MSCVAE_H = {"window_sizes": [6, 24, 168], "seq_len": 6, "stride": 2, "epochs": 15, "batch_size": 128, "val_fraction": 0.2, "q": 0.001}

HOURLY = [
    mspc_config("hourly", "hourly_pca_raw", "raw", "pca", label="PCA (raw)", **PCA),
    mspc_config("hourly", "hourly_pca_weekly", "weekly_baseline", "pca", label="PCA (weekly-baseline resid.)", **PCA),
    mspc_config("hourly", "hourly_pca_weekly_kde", "weekly_baseline", "pca", label="PCA (weekly, KDE limits)", limit_method="kde", **PCA),
    mspc_config("hourly", "hourly_pca_weekly_spot", "weekly_baseline", "pca", label="PCA (weekly, SPOT limits)", limit_method="spot", **PCA),
    mspc_config("hourly", "hourly_dpca_weekly", "weekly_baseline", "dpca", label="DPCA (weekly-baseline resid.)", lags=2, **PCA),
    mspc_config("hourly", "hourly_rpca_weekly", "weekly_baseline", "rpca", label="RPCA (weekly-baseline resid.)",
                window=8760, update_every=168, guard=0, exclude_ratio=1_000_000_000, limit_method="kde", **PCA),
]
if RUN_MSCVAE:
    HOURLY += [
        mspc_config("hourly", "hourly_mscvae_weekly", "weekly_baseline", "mscvae", label="MSCVAE (weekly-baseline resid.)", **MSCVAE_H),
        mspc_config("hourly", "hourly_mscvae_raw", "raw", "mscvae", label="MSCVAE (raw)", **MSCVAE_H),
    ]
hourly = run_all(HOURLY)

### Detector comparison
Own limits (FAR on normal periods away from events and outside the reference), then the same
detectors with the threshold moved to a 5% false-alarm rate.

In [ ]:
summary_table(hourly)

In [ ]:
matched_table(HOURLY, hourly)

In [ ]:
fig = P.detector_tradeoff(pd.DataFrame([r["summary"] for r in hourly.values()]), "Hourly detectors: false alarms vs detected events")

### SIN blackout of 15/08/2023 (shaded)
Statistic / control limit (log scale); dots are persistent alarms.

In [ ]:
H = lambda n: hourly[n]["scores"]  # noqa: E731
panels = [(H("hourly_pca_weekly_kde"), "T2", "PCA T² (weekly-baseline resid., KDE limit)"),
          (H("hourly_pca_weekly_kde"), "SPE", "PCA SPE (weekly-baseline resid., KDE limit)"),
          (H("hourly_rpca_weekly"), "T2", "RPCA T² (weekly-baseline resid.)")]
if RUN_MSCVAE:
    panels.append((H("hourly_mscvae_weekly"), "phi", "MSCVAE φ (weekly-baseline resid., SPOT limit)"))
fig = P.control_chart(panels, "2023-08-12", "2023-08-19", ("2023-08-15 08:30", "2023-08-15 15:00"), "Fast anomaly: SIN blackout of 15/08/2023")

In [ ]:
fig = P.control_chart(panels, "2018-03-18", "2018-03-25", ("2018-03-21 15:48", "2018-03-21 21:00"), "Fast anomaly: N/NE blackout of 21/03/2018")

### Diagnosis: which variables explain the alarm?
PCA: reconstruction-based contributions (RBC, T² + SPE, each normalized by its limit) and complete
decomposition / partial contributions (CDC); MSCVAE: the author's root-cause analysis.

In [ ]:
def cdc(name, event_id):
    r = hourly[name]
    w = r["events"].set_index("event_id").loc[event_id]
    idx = r["scores"].index
    sel = (idx >= w.start) & (idx <= w.end) & r["scores"]["ratio"].notna().to_numpy()
    mask = sel & r["alarm"].to_numpy() if (sel & r["alarm"].to_numpy()).any() else sel
    return r["detector"].contributions(r["data"], mask, kind="cdc").assign(event_id=event_id)


for ev in ("blackout_2023_08_15", "blackout_2018_03_21"):
    items = [(hourly["hourly_pca_weekly_kde"]["contributions"], "PCA — RBC (T² + SPE)"), (cdc("hourly_pca_weekly_kde", ev), "PCA — CDC / partial")]
    if RUN_MSCVAE:
        items.append((hourly["hourly_mscvae_weekly"]["contributions"], "MSCVAE — RCA"))
    fig = P.contributions(items, ev, f"Who explains the {ev[9:].replace('_', '/')} alarm?")

In [ ]:
ev = hourly["hourly_pca_weekly_kde"]["events"]
ev[ev.kind == "blackout"][["event_id", "subsystems", "detected", "delay_h", "max_ratio", "alarm_share", "top_contributors"]]

In [ ]:
ev[ev.kind != "blackout"].sort_values("start", ascending=False).head(20)[["event_id", "subsystems", "detected", "delay_h", "max_ratio", "top_contributors"]]

### Alarms over the years (energy-transition drift)

In [ ]:
items = [(H("hourly_pca_weekly_kde"), "PCA (KDE)"), (H("hourly_rpca_weekly"), "RPCA")]
if RUN_MSCVAE:
    items.append((H("hourly_mscvae_weekly"), "MSCVAE"))
fig = P.far_by_year(items, "Hourly detectors: share of hours in alarm per year")

## Daily scale
Preprocessing: `raw` or `calendar` residuals (OLS on the reference with trend, Fourier terms, day of
week and holidays). Static references age quickly with the wind/solar growth; the moving-window PCA
re-fits every 30 days on the last 2 years, excluding strongly alarmed days (bounded guard).

In [ ]:
MSCVAE_D = {"window_sizes": [7, 30, 90], "seq_len": 5, "stride": 1, "epochs": 60, "batch_size": 64, "val_fraction": 0.2, "q": 0.001}

DAILY = [
    mspc_config("daily", "daily_pca_raw", "raw", "pca", label="PCA (raw)", **PCA),
    mspc_config("daily", "daily_pca_calendar", "calendar", "pca", label="PCA (calendar resid.)", **PCA),
    mspc_config("daily", "daily_pca_calendar_kde", "calendar", "pca", label="PCA (calendar, KDE limits)", limit_method="kde", **PCA),
    mspc_config("daily", "daily_pca_calendar_spot", "calendar", "pca", label="PCA (calendar, SPOT limits)", limit_method="spot", **PCA),
    mspc_config("daily", "daily_dpca_calendar", "calendar", "dpca", label="DPCA (calendar resid.)", lags=1, **PCA),
    mspc_config("daily", "daily_rpca_calendar", "calendar", "rpca", label="RPCA (calendar resid.)",
                window=730, update_every=30, guard=7, accept_after=90, exclude_ratio=3.0, limit_method="kde", **PCA),
]
if RUN_MSCVAE:
    DAILY += [
        mspc_config("daily", "daily_mscvae_calendar", "calendar", "mscvae", label="MSCVAE (calendar resid.)", **MSCVAE_D),
        mspc_config("daily", "daily_mscvae_raw", "raw", "mscvae", label="MSCVAE (raw)", **MSCVAE_D),
    ]
daily = run_all(DAILY)
summary_table(daily)

In [ ]:
matched_table(DAILY, daily)

### 2021 water crisis (shaded) vs SE/CO stored energy

In [ ]:
ear = daily_matrix("2019-01-01", None)["SE_ear"]
fig = P.daily_crisis(daily["daily_rpca_calendar"]["scores"], "RPCA (calendar resid.)", ear, "2019-06-01", "2023-06-30", ("2021-06-01", "2021-11-30"))

In [ ]:
fig = P.far_by_year([(daily[n]["scores"], daily[n]["summary"]["detector"]) for n in ("daily_pca_calendar_kde", "daily_rpca_calendar")],
                    "Daily detectors: share of days in alarm per year")

In [ ]:
dev = daily["daily_rpca_calendar"]["events"]
dev[["event_id", "kind", "start", "end", "detected", "delay_h", "alarm_share", "top_contributors"]]